### v2 Pipeline — Sparkov Dataset

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent  # new code/ -> code/ -> repo root
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / "src" / "newDataScripts"))

os.makedirs(r"c:\DS440\FinancialBehaviorXAI\data\v2", exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Scripts path exists?", (PROJECT_ROOT / "src" / "newDataScripts").exists())

### Data Preprocessing

In [ ]:
from data_preprocessing import preprocess_data

df = preprocess_data(r"c:\DS440\FinancialBehaviorXAI\data\newDataTest\TrainingDataset.csv")
df.to_csv(r"c:\DS440\FinancialBehaviorXAI\data\v2\preprocessed.csv", index=False)
print(df.shape)
df.head()

### Monthly Spending Features

In [ ]:
from monthly_spending import calculate_monthly_spending_features

df = calculate_monthly_spending_features(df)
df.to_csv(r"c:\DS440\FinancialBehaviorXAI\data\v2\monthly_spending.csv", index=False)
print(df.shape)
df.head()

### Trends and Volatility

In [ ]:
from trends import calculate_trends_and_volatility

df = calculate_trends_and_volatility(df)
df.to_csv(r"c:\DS440\FinancialBehaviorXAI\data\v2\trends.csv", index=False)
print(df.shape)
df.head()

### Frequency and Regularity

In [ ]:
from frequency_regularity import calculate_frequency_regularity

df = calculate_frequency_regularity(df)
df.to_csv(r"c:\DS440\FinancialBehaviorXAI\data\v2\frequency_regularity.csv", index=False)
print(df.shape)
df.head()

### Habit Indicators

In [ ]:
from habit_indicators import calculate_habit_indicators

df = calculate_habit_indicators(df)
df.to_csv(r"c:\DS440\FinancialBehaviorXAI\data\v2\habit_indicators.csv", index=False)
print(df.shape)
df.head()

### Overspending Target Creation

In [ ]:
from target_creation import create_overspending_target

df = create_overspending_target(df)
df.to_csv(r"c:\DS440\FinancialBehaviorXAI\data\v2\model_ready.csv", index=False)
print(df.shape)
print(df["overspend_target_t1"].value_counts(normalize=True))
df.head()

### Sanity Check

In [ ]:
na_counts = df.isna().sum()
print("Columns with NAs:")
print(na_counts[na_counts > 0])
print("
Target balance:")
print(df["overspend_target_t1"].value_counts(normalize=True))

### Modeling — Setup

In [ ]:
import importlib
import modeling
importlib.reload(modeling)
from modeling import train_and_compare_models

### Modeling — Default Class Weight {0:1, 1:2}

In [ ]:
results_balanced, models_balanced, split_balanced = train_and_compare_models(df)
print(results_balanced)

### Modeling — Mild Class Weight {0:1, 1:1.5}

In [ ]:
results_mild, models_mild, split_mild = train_and_compare_models(
    df, class_weight={0: 1, 1: 1.5}
)
print(results_mild)

### Modeling — Hyperparameter Tuning

In [ ]:
results_tuned, models_tuned, split_tuned = train_and_compare_models(
    df, tune=True, n_trials=50
)
print(results_tuned)

### Modeling — Raw Features Only (Ablation)

In [ ]:
raw_cols = [
    "year", "month", "entertainment", "food_dining", "gas_transport",
    "grocery_net", "grocery_pos", "health_fitness", "home", "kids_pets",
    "misc_net", "misc_pos", "personal_care", "shopping_net", "shopping_pos",
    "travel", "total_amount", "median_monthly_income",
    "period", "first", "last", "overspend_current_month", "overspend_target_t1"
]
df_raw = df[raw_cols].copy()
results_raw, models_raw, split_raw = train_and_compare_models(df_raw)
print("RAW FEATURES:")
print(results_raw)
print("
ENGINEERED FEATURES:")
print(results_tuned)

### Precision-Recall Threshold Analysis

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np

best_model = models_raw["gradient_boosting"]
X_test = split_raw["X_test"]
y_test = split_raw["y_test"]

y_score = best_model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_score)

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.axvline(x=0.5, color="gray", linestyle="--", label="Default threshold (0.5)")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision vs Recall across thresholds")
plt.legend()
plt.grid(True)
plt.show()

f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx = f1_scores.argmax()
print(f"Best F1 threshold: {thresholds[best_idx]:.3f}")
print(f"  Precision: {precisions[best_idx]:.3f}")
print(f"  Recall:    {recalls[best_idx]:.3f}")
print(f"  F1:        {f1_scores[best_idx]:.3f}")

### SHAP Feature Importance

In [ ]:
import shap
import numpy as np

best_model = models_raw["gradient_boosting"]
X_test = split_raw["X_test"]
y_test = split_raw["y_test"]

preprocessor = best_model.named_steps["preprocessor"]
X_test_transformed = preprocessor.transform(X_test)

numeric_cols = X_test.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

if categorical_cols:
    ohe_feature_names = preprocessor.named_transformers_["cat"]["onehot"].get_feature_names_out(categorical_cols).tolist()
    all_feature_names = numeric_cols + ohe_feature_names
else:
    all_feature_names = numeric_cols

explainer = shap.TreeExplainer(best_model.named_steps["model"])
shap_values = explainer.shap_values(X_test_transformed)

shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=all_feature_names,
    max_display=20
)